# RFModelEva — Evaluation and Discussion (Random Forest)

## Setup

In [1]:
import joblib
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from evaluation_utils import (
    load_yelp_split,
    print_evaluation,
    print_top_terms,
    validate_pipeline_labels,
)

PART_B_DIR = Path.cwd().parents[0]
DATA_FILE = PART_B_DIR / "data" / "yelp_clean.csv"
MODEL_FILE = PART_B_DIR / "models" / "rf_tuned.joblib"        # tuned model from Q3
BASELINE_FILE = PART_B_DIR / "models" / "rf_pipeline.joblib"  # untuned model from Q2

## Load tuned model

In [2]:
# --- Reload the tuned Random Forest and check it carries the expected 3-class labels ---
pipeline = joblib.load(MODEL_FILE)
validate_pipeline_labels(pipeline, "Random Forest")
print("Classes:", list(pipeline.named_steps["clf"].classes_))

Classes: ['negative', 'neutral', 'positive']


## Tuned model evaluation

In [3]:
# --- Recreate the same held-out test set used throughout Part B ---
df, X_train, X_test, y_train, y_test = load_yelp_split(DATA_FILE)

print_top_terms(pipeline, X_train, y_train, "Random Forest")
print_evaluation(pipeline, X_test, y_test, "Random Forest")

=== Top Terms Per Sentiment Class (Random Forest) ===

negative:
horrible, rude, bad, terrible, tell, no, manager, customer, not even, ask

neutral:
three star, decent, not bad, pretty, pretty good, okay, ok, average, bit, good

positive:
great, love, delicious, amaze, awesome, favorite, highly recommend, excellent, perfect, best

=== Q4: Tuned Yelp Random Forest 3-Class Classification Report ===
              precision    recall  f1-score   support

    negative     0.7619    0.8013    0.7811      2400
     neutral     0.4961    0.3675    0.4222      1200
    positive     0.7514    0.8100    0.7796      2400

    accuracy                         0.7180      6000
   macro avg     0.6698    0.6596    0.6610      6000
weighted avg     0.7045    0.7180    0.7087      6000

=== Q4: Tuned Yelp Random Forest Summary ===
Accuracy:        0.7180
Macro Precision: 0.6698
Macro Recall:    0.6596
Macro F1-score:  0.6610
Weighted F1:     0.7087

=== Q4: Tuned Yelp Random Forest Confusion Matrix ===

## Untuned vs tuned

In [4]:
# --- Side-by-side metrics for the hyperparameter discussion: what tuning bought us ---
def score_row(model, name):
    preds = model.predict(X_test)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_test, preds, average="macro", zero_division=0
    )
    weighted_f1 = precision_recall_fscore_support(
        y_test, preds, average="weighted", zero_division=0
    )[2]
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Macro P": macro_p,
        "Macro R": macro_r,
        "Macro F1": macro_f1,
        "Weighted F1": weighted_f1,
    }


comparison = pd.DataFrame([
    score_row(joblib.load(BASELINE_FILE), "Untuned baseline (Q2)"),
    score_row(pipeline, "Tuned (Q3)"),
]).set_index("Model").round(4)

print("=== Q4: Random Forest - Untuned vs Tuned ===")
print(comparison.to_string())

gain = comparison.loc["Tuned (Q3)", "Macro F1"] - comparison.loc["Untuned baseline (Q2)", "Macro F1"]
print(f"\nMacro F1 gain from hyperparameter tuning: {gain:+.4f}")

=== Q4: Random Forest - Untuned vs Tuned ===
                       Accuracy  Macro P  Macro R  Macro F1  Weighted F1
Model                                                                   
Untuned baseline (Q2)    0.7078   0.6852   0.5957    0.5445       0.6402
Tuned (Q3)               0.7180   0.6698   0.6596    0.6610       0.7087

Macro F1 gain from hyperparameter tuning: +0.1165
